# 黑马头条项目｜第03天：自媒体文章发布

> 目标：搭建自媒体前后端链路，完成素材上传与分页查询、频道查询、自媒体文章查询，以及文章保存/发布时的素材关系维护。

本篇根据百度网盘第03天“自媒体文章发布”讲义整理，源码和命令只作静态参照。未启动 Nacos、MySQL、MinIO、Nginx、微服务或前端项目，未调用接口。

## 学习地图

```text
自媒体前端 wemedia-web
        │  /wemedia/**
        ▼
自媒体网关（解析 JWT，转发 userId）
        │
        ▼
leadnews-wemedia
  ├─ 素材：上传、列表、收藏筛选
  ├─ 频道：查询所有频道
  └─ 文章：分页查询、保存草稿、提交发布
        │
        ├─ MySQL：wm_material / wm_channel / wm_news / wm_news_material
        └─ MinIO：图片对象
```

本日的关键设计是：网关解析登录信息后把用户标识放进请求头，微服务拦截器再把它放进当前请求线程的 ThreadLocal，业务层据此隔离不同自媒体用户的数据。

## 1. 自媒体前后端搭建

### 1.1 后端模块

1. 从资料中导入 `heima-leadnews-wemedia`，加入 `heima-leadnews-service` 的 Maven 子模块。
2. 执行自媒体数据库脚本，创建 `leadnews_wemedia` 及相关表。
3. 在配置中心增加数据源和 MyBatis-Plus 别名包配置，示例中的数据库账号、密码、地址统一改为环境变量或密钥配置。
4. 导入 `heima-leadnews-wemedia-gateway`，在网关增加：

```yaml
- id: wemedia
  uri: lb://leadnews-wemedia
  predicates:
    - Path=/wemedia/**
  filters:
    - StripPrefix=1
```

### 1.2 前端与 Nginx

自媒体前端使用 `wemedia-web`。讲义通过 Nginx 虚拟主机同时代理静态页面和网关请求：页面访问一个端口，`/wemedia/MEDIA/**` 代理到自媒体网关。实际部署时应把网关地址、前端目录和端口放入配置，并限制跨域来源。

联调顺序：启动 Nginx、自媒体微服务、自媒体网关，再从前端验证登录。

## 2. 用户身份在网关与微服务之间传递

### 2.1 网关写入用户标识

网关过滤器完成 JWT 解析后，从 claims 取出用户 ID，写入下游请求头 `userId`。这里的请求头只承载已经验证过的内部身份，不应允许客户端直接伪造后绕过网关。

```java
Object userId = claimsBody.get("id");
ServerHttpRequest request = exchange.getRequest().mutate()
    .headers(headers -> headers.add("userId", String.valueOf(userId)))
    .build();
return chain.filter(exchange.mutate().request(request).build());
```

### 2.2 微服务拦截器与 ThreadLocal

`WmTokenInterceptor` 在 `preHandle` 读取 `userId`，构造最小化的 `WmUser` 放入 ThreadLocal；请求结束后必须清理，避免线程池复用造成串号。

```java
public final class WmThreadLocalUtil {
    private static final ThreadLocal<WmUser> HOLDER = new ThreadLocal<>();
    public static void setUser(WmUser user) { HOLDER.set(user); }
    public static WmUser getUser() { return HOLDER.get(); }
    public static void clear() { HOLDER.remove(); }
}

@Override
public void postHandle(HttpServletRequest req, HttpServletResponse resp,
                       Object handler, ModelAndView view) {
    WmThreadLocalUtil.clear();
}
```

讲义源码中出现过 `WmThreadLocalUtil` 与 `WmThreadLocalUtils` 两种命名，落地时要统一类名，并在异常路径也保证清理。

## 3. 素材管理：图片上传

### 3.1 表结构与接口

`wm_material` 保存自媒体素材：`id`、`user_id`、`url`、`type`（0 图片、1 视频）、`is_collection`、`created_time`。

接口：

| 项目 | 内容 |
|---|---|
| 路径 | `POST /api/v1/material/upload_picture` |
| 参数 | `MultipartFile` |
| 成功结果 | `ResponseResult`，返回素材对象和图片 URL |
| 失败结果 | 参数无效或上传失败 |

### 3.2 上传处理流程

1. 校验文件为空和大小。
2. 生成随机文件名，保留安全的扩展名。
3. 通过 `FileStorageService.uploadImgFile` 写入 MinIO。
4. 取 ThreadLocal 中的用户 ID，保存 `WmMaterial`。
5. 返回素材记录，让前端回显图片并在文章编辑器中引用 URL。

```java
if (file == null || file.isEmpty()) {
    return ResponseResult.errorResult(AppHttpCodeEnum.PARAM_INVALID);
}
String key = UUID.randomUUID().toString().replace("-", "") + suffix;
String url = fileStorageService.uploadImgFile("", key, file.getInputStream());
WmMaterial material = new WmMaterial();
material.setUserId(WmThreadLocalUtil.getUser().getId());
material.setUrl(url);
material.setType((short) 0);
material.setIsCollection((short) 0);
material.setCreatedTime(new Date());
save(material);
```

生产实现还应检查 MIME 类型、文件大小、扩展名和内容签名，并在对象上传成功而数据库写入失败时设计补偿清理。

## 4. 素材列表查询与分页

### 4.1 接口与 DTO

`POST /api/v1/material/list`，参数 `WmMaterialDto extends PageRequestDto`，其中 `isCollection` 用于筛选收藏素材。

返回分页结构包含 `data`、`currentPage`、`size`、`total`。

### 4.2 查询条件

Service 负责：

- 调用 `dto.checkParam()` 校验分页参数；
- 以 ThreadLocal 用户 ID 限制数据范围；
- `isCollection = 1` 时增加收藏条件；
- 按 `created_time` 倒序；
- 用 MyBatis-Plus `Page` 执行分页并封装 `PageResponseResult`。

```java
IPage<WmMaterial> page = new Page<>(dto.getPage(), dto.getSize());
LambdaQueryWrapper<WmMaterial> q = new LambdaQueryWrapper<>();
if (Short.valueOf((short) 1).equals(dto.getIsCollection())) {
    q.eq(WmMaterial::getIsCollection, (short) 1);
}
q.eq(WmMaterial::getUserId, WmThreadLocalUtil.getUser().getId())
 .orderByDesc(WmMaterial::getCreatedTime);
page = page(page, q);
```

启动类要注册 `PaginationInnerInterceptor(DbType.MYSQL)`，否则分页可能退化为全量查询。

## 5. 频道查询

### 5.1 数据与接口

`wm_channel` 保存频道名称、描述、是否默认、是否启用、排序值和创建时间。

接口：`GET /api/v1/channel/channels`，无参数，返回所有频道。

### 5.2 分层实现

- `WmChannelMapper extends BaseMapper<WmChannel>`；
- `WmChannelService extends IService<WmChannel>`；
- Service 调用 `list()`，Controller 调用 Service 并返回 `ResponseResult`。

```java
@GetMapping("/channels")
public ResponseResult findAll() {
    return wmChannelService.findAll();
}

@Override
public ResponseResult findAll() {
    return ResponseResult.okResult(list());
}
```

实际产品通常还应按 `status` 过滤禁用频道，并按 `ord` 排序；讲义示例强调的是最小可用链路。

## 6. 自媒体文章列表

### 6.1 `wm_news` 核心字段

文章表保存：用户、标题、JSON 内容、布局类型、频道、标签、创建/提交时间、状态、定时发布时间、拒绝原因、发布库文章 ID、封面图片和上下架标志。

状态枚举：

| 值 | 含义 |
|---:|---|
| 0 | 草稿 |
| 1 | 已提交，待审核 |
| 2 | 审核失败 |
| 3 | 人工审核中 |
| 4 | 人工审核通过 |
| 8 | 审核通过，待发布 |
| 9 | 已发布 |

### 6.2 查询接口

`POST /api/v1/news/list`，参数 `WmNewsPageReqDto`：分页参数、状态、开始/结束发布时间、频道 ID、标题关键字。

Service 查询顺序：校验参数 → 获取当前用户 → 拼接状态/频道/时间范围/标题模糊条件 → 限定 `user_id` → 按创建时间倒序 → 返回分页结果。

```java
if (dto.getStatus() != null) q.eq(WmNews::getStatus, dto.getStatus());
if (dto.getChannelId() != null) q.eq(WmNews::getChannelId, dto.getChannelId());
if (dto.getBeginPubDate() != null && dto.getEndPubDate() != null) {
    q.between(WmNews::getPublishTime,
              dto.getBeginPubDate(), dto.getEndPubDate());
}
if (StringUtils.isNotBlank(dto.getKeyword())) {
    q.like(WmNews::getTitle, dto.getKeyword());
}
q.eq(WmNews::getUserId, user.getId())
 .orderByDesc(WmNews::getCreatedTime);
```

如果用户信息为空，返回需要登录；不能把用户 ID 从前端请求体直接当作可信条件。

## 7. 保存草稿与提交文章

### 7.1 关联表

提交文章除了写入 `wm_news`，还要维护 `wm_news_material`：

- `material_id`：素材 ID；
- `news_id`：文章 ID；
- `type`：0 内容图片引用，1 封面引用；
- `ord`：在内容或封面列表中的顺序。

文章提交或修改的事务流程：

```text
校验 DTO
  ├─ 无 id：新增 wm_news
  └─ 有 id：删除旧的 wm_news_material，再更新 wm_news
若为草稿：结束
解析 content 中的 image 节点
校验这些 URL 都属于当前用户的素材
批量保存内容图片关系
根据封面类型保存封面关系
```

### 7.2 请求 DTO

```java
@Data
public class WmNewsDto {
    private Integer id;
    private String title;
    private Integer channelId;
    private String labels;
    private Date publishTime;
    private String content;       // JSON 数组字符串
    private Short type;           // 0 无图、1 单图、3 多图、-1 自动
    private Date submitedTime;
    private Short status;         // 0 草稿、1 提交
    private List<String> images;  // 封面 URL 列表
}
```

讲义给出的提交路径是 `/api/v1/channel/submit`，实现时应以接口文档和前端调用保持一致，通常会将文章提交归入 `/api/v1/news/**`。

## 8. 文章素材关系的实现

### 8.1 批量插入 Mapper

```java
void saveRelations(@Param("materialIds") List<Integer> materialIds,
                   @Param("newsId") Integer newsId,
                   @Param("type") Short type);
```

```xml
<insert id="saveRelations">
  INSERT INTO wm_news_material(material_id, news_id, type, ord)
  VALUES
  <foreach collection="materialIds" index="ord" item="mid" separator=",">
    (#{mid}, #{newsId}, #{type}, #{ord})
  </foreach>
</insert>
```

### 8.2 从内容 JSON 提取图片

文章内容是数组，每个元素通常包含 `type` 和 `value`。只提取 `type = image` 的 URL，再到 `wm_material` 中按 URL 查询素材 ID。

```java
List<Map> nodes = JSON.parseArray(dto.getContent(), Map.class);
List<String> urls = nodes.stream()
    .filter(node -> "image".equals(node.get("type")))
    .map(node -> (String) node.get("value"))
    .collect(Collectors.toList());
```

如果素材不存在、数量对不上，抛出业务异常让事务回滚，避免文章保存成功但留下无效引用。

## 9. 封面类型与自动匹配

常量：

```java
WM_NEWS_NONE_IMAGE = 0;
WM_NEWS_SINGLE_IMAGE = 1;
WM_NEWS_MANY_IMAGE = 3;
WM_NEWS_TYPE_AUTO = -1;
WM_CONTENT_REFERENCE = 0;
WM_COVER_REFERENCE = 1;
```

当 DTO 的封面类型为自动：

- 内容图片数为 0：文章类型设为无图；
- 内容图片数为 1～2：取 1 张作为单图封面；
- 内容图片数不少于 3：取前 3 张作为多图封面。

手工选择封面时，直接校验并保存 DTO 中的图片 URL。自动匹配后要回写 `wm_news.type` 和 `wm_news.images`，再以 `WM_COVER_REFERENCE` 写入关联表。

## 10. 文章保存 Service 的骨架

```java
@Transactional
public ResponseResult submitNews(WmNewsDto dto) {
    if (dto == null || dto.getContent() == null) {
        return ResponseResult.errorResult(AppHttpCodeEnum.PARAM_INVALID);
    }

    WmNews news = new WmNews();
    BeanUtils.copyProperties(dto, news);
    news.setImages(StringUtils.join(dto.getImages(), ","));
    if (WemediaConstants.WM_NEWS_TYPE_AUTO.equals(dto.getType())) {
        news.setType(null);
    }
    saveOrUpdateNews(news); // 补用户、创建/提交时间、enable，并处理旧关系

    if (WmNews.Status.NORMAL.getCode() == dto.getStatus()) {
        return ResponseResult.okResult(AppHttpCodeEnum.SUCCESS);
    }

    List<String> contentUrls = extractImageUrls(dto.getContent());
    saveRelationsByUrls(contentUrls, news.getId(), WM_CONTENT_REFERENCE);
    saveCoverRelations(dto, news, contentUrls);
    return ResponseResult.okResult(AppHttpCodeEnum.SUCCESS);
}
```

`saveOrUpdateNews` 在新增时插入文章，在修改时先删除旧素材关系再更新文章。文章、关系和素材校验必须放在同一事务边界内；异常时由全局异常处理器转换为统一错误响应。

## 11. 接口与测试清单

| 功能 | 方法 | 路径 |
|---|---|---|
| 图片上传 | POST | `/api/v1/material/upload_picture` |
| 素材分页 | POST | `/api/v1/material/list` |
| 查询频道 | GET | `/api/v1/channel/channels` |
| 查询文章 | POST | `/api/v1/news/list` |
| 保存/提交文章 | POST | 以讲义接口文档为准，示例为 `/api/v1/channel/submit` |

建议按以下顺序联调：登录并验证网关传递用户 ID → 上传图片并确认素材记录 → 查询素材列表 → 查询频道 → 保存草稿 → 提交带图片文章 → 修改文章并确认旧关系被删除 → 提交无效素材并确认事务回滚。

本篇资料中的固定 IP、账号、密码、端口和本地路径均为演示配置，笔记已替换为占位符。

## 12. 复习要点

- 网关负责认证和路由，业务微服务从 ThreadLocal 取得当前用户；请求结束必须清理。
- MinIO 保存图片二进制，`wm_material` 保存图片 URL 与业务归属，两者需要补偿策略。
- 分页查询必须叠加当前用户条件；分页插件要显式配置。
- 文章内容图片和封面图片都要通过素材表校验，关联表用 `type` 区分引用位置。
- 自动封面由内容图片数量决定，修改文章先清理旧关联再写新关联。
- `@Transactional` 只能覆盖数据库事务，MinIO 对象上传要额外考虑失败补偿和孤儿对象。

### 自测题

1. 为什么不能只把 `userId` 放在前端请求体中？
2. 素材上传成功但文章保存失败时，如何清理 MinIO 中的对象？
3. 修改文章时为什么要先删除 `wm_news_material` 旧关系？
4. 如何保证同一张图片既作为正文图片又作为封面时，关联顺序和类型都正确？
5. 文章列表查询的状态、频道、日期、关键字条件应如何组合？